# Regex-Only PII Evaluation

Evaluate `CustomPatternRecognizer` by itself, without loading a spaCy, transformers, underthesea, or other NER model.

In [ ]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
from datasets import load_dataset

from src.pipeline.Evaluator import PIIEvaluator
from src.pipeline.Recognizers.CustomPatternRecognizer import CustomPatternRecognizer
from src.pipeline.Utils import display_evaluation_results, with_prefix

## Load Evaluation Data

In [ ]:
from src.pipeline.Utils import load_hf_token
# Choose dataset split to test: "train", "val", "test", or "all".
split_choice = "test"

# Keep this small for quick iteration. Set to None to use the full selected split(s).
sample_size = 5000

dataset_name = "nguyenlamtung/pii-masking-95k-preencoded"
print(f"Loading dataset: {dataset_name}")
dataset = load_dataset(dataset_name, token=load_hf_token())

available_splits = list(dataset.keys())
print("Available splits in dataset:", available_splits)

if split_choice == "all":
    selected_splits = available_splits
else:
    mapped_split = "validation" if split_choice == "val" and "validation" in dataset else split_choice
    if mapped_split in dataset:
        selected_splits = [mapped_split]
    else:
        print(f"Split '{split_choice}' not found, falling back to 'train'.")
        selected_splits = ["train"]

dfs = []
for split in selected_splits:
    split_df = dataset[split].to_pandas()
    if sample_size is not None and len(split_df) > sample_size:
        split_df = split_df.sample(n=sample_size, random_state=42)
    split_df["split"] = split
    dfs.append(split_df)

df_eval = pd.concat(dfs, ignore_index=True)

if "text" in df_eval.columns and "source_text" not in df_eval.columns:
    df_eval["source_text"] = df_eval["text"]

print(f"Loaded and sampled {len(df_eval)} rows across splits: {selected_splits}")
df_eval.head()

## Regex-Only Analyzer

In [ ]:
class RegexOnlyAnalyzer:
    """Minimal analyzer interface for PIIEvaluator using only regex recognizers."""

    def __init__(self, recognizer=None):
        self.patterns = recognizer or CustomPatternRecognizer()
        self.patterns.load_model()

    def analyze(self, text, language="vi", score_threshold=0.0):
        results = []
        for recognizer in self.patterns.recognizers:
            if recognizer.supported_language != language:
                continue

            results.extend(
                recognizer.analyze(
                    text=text,
                    entities=recognizer.supported_entities,
                    nlp_artifacts=None,
                )
            )

        return [result for result in results if result.score >= score_threshold]


regex_analyzer = RegexOnlyAnalyzer()
sample_text = "Email toi la test@example.com, sdt 0912345678, stk 123456789012."
[(r.entity_type, sample_text[r.start:r.end], r.score) for r in regex_analyzer.analyze(sample_text, language="vi")]

## Evaluate

In [ ]:
evaluator = PIIEvaluator()

overall, per_entity = evaluator.evaluate_presidio(
    df_eval=df_eval,
    analyzer=regex_analyzer,
    language="vi",
    score_threshold=0.0,
    use_type_mapping=True,
    return_per_entity=True,
)

print("Regex-Only Metrics:")
for key, value in overall.items():
    print(f"- {key}: {value}")

print("\nRegex-Only Per-Entity Metrics:")
for entity, metrics in per_entity.items():
    print(f"- {entity}: Precision={metrics['precision']:.4f}, Recall={metrics['recall']:.4f}, F1={metrics['f1']:.4f}")

## Display Results

In [ ]:
overall_row = with_prefix("mapped_types", overall)
overall_row["step_name"] = "Regex Only"
overall_df = pd.DataFrame([overall_row])

per_entity_df = pd.DataFrame.from_dict(per_entity, orient="index").reset_index().rename(columns={"index": "entity_type"})
per_entity_df["step_name"] = "Regex Only"

display_evaluation_results(overall_df, per_entity_df)